In [ ]:
# INSTALL LIBRARIES
!pip install transformers datasets scikit-learn matplotlib seaborn
!pip install transformers==4.30.2 datasets==2.14.5 torch --no-cache-dir

In [ ]:
# IMPORT LIBRARIES
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# LOAD DATASET
from datasets import load_dataset

dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data = dataset["test"]

In [ ]:
# PREPROCESSING
def clean_text(example):
    example["text"] = example["text"].lower()
    return example

train_data = train_data.map(clean_text)
test_data = test_data.map(clean_text)

In [ ]:
# TOKENIZATION
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
# SPLIT TRAIN → TRAIN + VALIDATION
train_dataset = train_data.select(range(20000))
val_dataset = train_data.select(range(20000, 25000))
test_dataset = test_data.select(range(5000))

train_dataset = train_data.select(range(800))
val_dataset = train_data.select(range(800, 1000))
test_dataset = test_data.select(range(300))

In [ ]:
# LOAD MODEL
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

In [ ]:
# METRICS FUNCTION
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
!pip install transformers --upgrade

In [ ]:
# TRAINING ARGUMENTS
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    logging_steps=10,
    disable_tqdm=True   # IMPORTANT
)

In [ ]:
# TRAIN MODEL
from transformers.utils import logging
logging.disable_progress_bar()
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
# EVALUATION
trainer.train()

preds = trainer.predict(test_dataset)

In [ ]:
# CONFUSION MATRIX
preds = trainer.predict(test_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# EXPERIMENT 1 (FREEZE BERT)
for param in model.distilbert.parameters():
    param.requires_grad = False

In [ ]:
# EXPERIMENT 2 (FINE-TUNE LAST LAYERS)
for param in model.distilbert.transformer.layer[-2:].parameters():
    param.requires_grad = True

Experiment Results:

1. Full fine-tuning → Best accuracy
2. Frozen BERT → Lower performance
3. Partial fine-tuning → Balanced performance

Conclusion:
Fine-tuning improves model performance significantly.